# CALM Usage

## Imports

In [ ]:
from calm import CALMRegressor

import numpy as np
import effector
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import os
import tensorflow as tf
import random

## Configuration

In [2]:
def set_random_seeds(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
            
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
set_random_seeds(42)

## Synthetic Dataset

For this example, the response variable has once again an interaction between two features in the form of regions. Here, however, we take the complexity of the model one step further, by making the contribution of feature $x_3$ have 4 different branches instead of two. Which branch it follows now depends on 2 features, instead of 1.

Specifically, the response variable is generated by the formula:
$$
y = x_1^2 + \log(|x_2|) + 2 
\begin{cases}
\sin(\frac{\pi}{2} x_3) & x_1 \geq 0, x_2 \geq 0 \\
\cos(\frac{\pi}{2} x_3) & x_1 \geq 0, x_2 < 0 \\
\sin(2 \pi x_3) & x_1 < 0, x_2 \geq 0 \\
\cos(2 \pi x_3) & x_1 < 0, x_2 < 0
\end{cases}
$$

The 3 variables are again sampled uniformly and independently from $[-1, 1]$, and we generate 1000 sample data points.

In [3]:
dataset = effector.datasets.IndependentUniform(dim=3, low=-1, high=1)
x = dataset.generate_data(1_000)
features = [f"x_{i+1}" for i in range(x.shape[1])]

In [4]:
class RegionalGenerator2(effector.models.Base):
    def __init__(self):
        super().__init__(name=self.__class__.__name__)

    def predict(self, x: np.ndarray) -> np.ndarray:
        y = x[:, 0]**2 + np.log(np.abs(x[:, 1])) + 2 * np.where(
            x[:, 0] >= 0,
            np.where(
                x[:, 1] >= 0,
                np.sin(np.pi * x[:, 2] / 2),
                np.cos(np.pi * x[:, 2] / 2),
            ),
            np.where(
                x[:, 1] >= 0,
                np.sin(2 * np.pi * x[:, 2]),
                np.cos(2 * np.pi * x[:, 2]),
            ),
        )
        return y

    def jacobian(self, x: np.ndarray) -> np.ndarray:
        y = np.zeros_like(x)
        y[:, 0] = 2 * x[:, 0]
        y[:, 1] = 1 / x[:, 1]
        y[:, 2] = 2 * np.where(
            x[:, 0] >= 0,
            np.where(
                x[:, 1] >= 0,
                np.cos(np.pi * x[:, 2] / 2) * np.pi / 2,
                -np.sin(np.pi * x[:, 2] / 2) * np.pi / 2,
            ),
            np.where(
                x[:, 1] >= 0,
                np.cos(2 * np.pi * x[:, 2]) * 2 * np.pi,
                -np.sin(2 * np.pi * x[:, 2]) * 2 * np.pi,
            ),
        )
        return y
model = RegionalGenerator2()
y = model.predict(x)

After generating the data, we perform the standard train-test split.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(x, y)

## CALM with default parameters (blackbox=XGB, region detector=PDP, masked GAM=EBM)

In [6]:
calm = CALMRegressor()
calm.fit(X_train, y_train, feat_labels=features)
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

y_pred = calm.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"CALM R^2 Score: {r2:.4f}")

100%|██████████| 3/3 [00:00<00:00, 14.59it/s]
/Users/dimitriskyriakopoulos/Documents/ath/effector-calm-test/effector/calm/calm-env/lib/python3.10/site-packages/interpret/glassbox/_ebm/_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


Feature: feature_0
🌳 Full Tree Structure:
───────────────────────
x_0 🔹 [id: 0 | heter: 0.81 | inst: 750 | w: 1.00]
    x_2 ≤ -0.70 🔹 [id: 1 | heter: 0.61 | inst: 110 | w: 0.15]
        x_1 ≤ 0.10 🔹 [id: 2 | heter: 0.20 | inst: 59 | w: 0.08]
        x_1 > 0.10 🔹 [id: 3 | heter: 0.07 | inst: 51 | w: 0.07]
    x_2 > -0.70 🔹 [id: 4 | heter: 0.61 | inst: 640 | w: 0.85]

Feature: feature_1
🌳 Full Tree Structure:
───────────────────────
x_1 🔹 [id: 0 | heter: 0.74 | inst: 750 | w: 1.00]

Feature: feature_2
🌳 Full Tree Structure:
───────────────────────
x_2 🔹 [id: 0 | heter: 0.97 | inst: 750 | w: 1.00]
    x_0 ≤ -0.00 🔹 [id: 1 | heter: 0.66 | inst: 382 | w: 0.51]
        x_1 ≤ 0.00 🔹 [id: 2 | heter: 0.11 | inst: 192 | w: 0.26]
        x_1 > 0.00 🔹 [id: 3 | heter: 0.14 | inst: 190 | w: 0.25]
    x_0 > -0.00 🔹 [id: 4 | heter: 0.53 | inst: 368 | w: 0.49]
        x_1 ≤ 0.00 🔹 [id: 5 | heter: 0.05 | inst: 173 | w: 0.23]
        x_1 > 0.00 🔹 [id: 6 | heter: 0.08 | inst: 195 | w: 0.26]

CALM R^2 Scor

## CALM setting supported parameters (blackbox=DNN, region detector=RHALE, masked GAM=NAM)

In [7]:
from calm.blackbox import KerasBlackBoxRegressor
from calm import RegionalRHALEDetector

blackbox = KerasBlackBoxRegressor # / RFRegressor / default: XGBRegressor

rhale_region_detector = RegionalRHALEDetector( # / default: RegionalPDPDetector
    heter_pcg_threshold=0.2,
    nof_splits_numerical=20,
)

masked_gam_name="MaskedNAMRegressor" # / "PyGAMRegressor" / "WithInteractionsEBMRegressor" / default: "NoInteractionsEBMRegressor"


calm = CALMRegressor(
    blackbox_model=KerasBlackBoxRegressor,
    region_detector=rhale_region_detector,
    masked_gam_name="MaskedNAMRegressor",
)

In [8]:
calm.fit(X_train, y_train, feat_labels=features)
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

y_pred = calm.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"CALM R^2 Score: {r2:.4f}")

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.1845 - mae: 1.4576  
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3.0830 - mae: 1.4363 
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.9946 - mae: 1.4171 
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.9186 - mae: 1.4002 
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.8529 - mae: 1.3840 
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7941 - mae: 1.3680 
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7403 - mae: 1.3533 
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.6893 - mae: 1.3388 
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.6393 - mae: 1.3239 
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.5899 - mae: 1.3086 
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.5407 - mae: 1.2928 
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.4913 - mae: 1.2767 
Epoch 13/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/ste

100%|██████████| 3/3 [00:00<00:00,  5.33it/s]
/Users/dimitriskyriakopoulos/Documents/ath/effector-calm-test/effector/calm/calm-env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.7392 
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1098 
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.5454
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2537 
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1442 
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0761 
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0192 
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9680 
Epoch 9/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.9237 
Epoch 10/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8821 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/stepWARNING:tensorflow:6 out of the last 12 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x30cddee60> triggered tf.function retracing. Tracing is expensive and the excessive nu

## CALM with custom parameters (blackbox=DNN, region detector=RHALE, masked GAM=EBM)

### Define a DNN blackbox

In [9]:
from calm.blackbox import BlackBoxModel
import tensorflow as tf
from tensorflow import keras


class KerasBlackBoxRegressor(BlackBoxModel):
    def __init__(
        self,
        input_dim=None,
        hidden_layers=[50, 50],
        activation="tanh", # changed relu to tanh
        learning_rate=0.001,
        verbose=1,
    ):
        self.model = keras.Sequential()

        if input_dim is not None:
            self.model.add(keras.layers.Input(shape=(input_dim,)))
        for units in hidden_layers:
            self.model.add(keras.layers.Dense(units, activation=activation))

        self.model.add(
            keras.layers.Dense(1, activation=None)
        )  # No activation for regression output

        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        self.model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

        self.verbose = verbose

    def fit(self, X, y, batch_size=200, epochs=200):
        self.model.fit(
            X,
            y,
            batch_size=batch_size,
            epochs=epochs,
            verbose=self.verbose,
        )

    def forward(self, X):
        return self.model(X).numpy().squeeze()

    def predict(self, X):
        return self.forward(X)

    def jac(self, X):
        x_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
        with tf.GradientTape() as t:
            t.watch(x_tensor)
            pred = self.model(x_tensor)
            grads = t.gradient(pred, x_tensor)
        return grads.numpy()

### Define the region detector

In [16]:
from calm import RegionDetector

class RegionalPDPDetector(RegionDetector):
    def __init__(
        self,
        heter_pcg_threshold,
        nof_splits_numerical,
        max_depth=2,
    ):
        self.heter_pcg_threshold = heter_pcg_threshold
        self.nof_splits_numerical = nof_splits_numerical
        self.max_depth = max_depth

    def detect_regions(
        self,
        X,
        blackbox_model: BlackBoxModel,
    ):
        self.regional_fe = effector.RegionalPDP(
            data=X,
            model=blackbox_model.forward,
            cat_limit=10,
            nof_instances="all",
        )
        self.regional_fe.fit(
            features="all",
            candidate_conditioning_features="all",
            space_partitioner=effector.space_partitioning.Best(
                min_heterogeneity_decrease_pcg=self.heter_pcg_threshold,
                numerical_features_grid_size=self.nof_splits_numerical,
                max_depth=self.max_depth,
            ),
        )

        tree = self.regional_fe.tree
        return tree

    
rhale_region_detector = RegionalPDPDetector(
    heter_pcg_threshold=0.3, # changed from 0.2 to 0.3
    nof_splits_numerical=20,
)

### Define the GAM

In [ ]:
### TO DO: let CALM receive a parameter of type MaskedGAM like the below class, 
### instead of only the parameter "masked_gam_name" of type str

# from interpret.glassbox import (
#     ExplainableBoostingRegressor,
# )

# from effector.calm.masked_fitting import MaskedGAM

# class NoInteractionsEBM(MaskedGAM):
#     def __init__(self):
#         super().__init__()

#     def fit(self, X, y, mask=None, axis_limits=None):
#         X = self._prepare_X(X)
#         mask = self._default_mask(X, mask)
#         if axis_limits is None:
#             self.axis_limits = np.array(
#                 [[np.min(X[:, i]), np.max(X[:, i])] for i in range(X.shape[1])]
#             ).T
#         else:
#             self.axis_limits = axis_limits

#         X_copy = X.copy()
#         X_copy[mask == 0] = np.nan  # Mask inactive features

#         self.model.fit(X_copy, y)  # Train the EBM model

#     def effect_unnorm(self, xs, i):
#         bins = self.model.bins_[i][0]
#         bin_idx = np.digitize(xs, bins) + 1
#         return self.model.term_scores_[i][bin_idx]

#     def predict(self, X, mask=None):
#         X = self._prepare_X(X)
#         mask = self._default_mask(X, mask)
#         X_copy = X.copy()
#         X_copy[mask == 0] = np.nan
#         return self.model.predict(X_copy)


# class NoInteractionsEBMRegressor(NoInteractionsEBM):
#     def __init__(self, **kwargs):
#         super().__init__()
#         self.model = ExplainableBoostingRegressor(
#             interactions=0, random_state=42, **kwargs
#         )

### Use calm with the defined blackbox, region detector and masked gam

In [17]:
calm = CALMRegressor(
    blackbox_model=KerasBlackBoxRegressor,
    region_detector=rhale_region_detector,
    masked_gam_name="PyGAMRegressor"
)
calm.fit(X_train, y_train, feat_labels=features)

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.0508 - mae: 1.4203  
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.8991 - mae: 1.3821 
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.8088 - mae: 1.3577 
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7657 - mae: 1.3433 
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7491 - mae: 1.3372 
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7404 - mae: 1.3355 
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7312 - mae: 1.3360 
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7222 - mae: 1.3378 
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7159 - mae: 1.3404 
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.7129 - mae: 1.3431 
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7121 - mae: 1.3448 
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7115 - mae: 1.3453 
Epoch 13/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/ste

100%|██████████| 3/3 [00:00<00:00, 43.95it/s]


,blackbox_model,<class '__mai...BoxRegressor'>
,blackbox_args,{}
,region_detector,<__main__.Reg...t 0x311891030>
,masked_gam_name,'PyGAMRegressor'
,masked_gam_args,{'dim': 6}
,refit_blackbox,True
,feat_types,None


In [18]:
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

Feature: feature_0
🌳 Full Tree Structure:
───────────────────────
x_0 🔹 [id: 0 | heter: 0.46 | inst: 750 | w: 1.00]

Feature: feature_1
🌳 Full Tree Structure:
───────────────────────
x_1 🔹 [id: 0 | heter: 0.39 | inst: 750 | w: 1.00]

Feature: feature_2
🌳 Full Tree Structure:
───────────────────────
x_2 🔹 [id: 0 | heter: 0.42 | inst: 750 | w: 1.00]
    x_0 ≤ -0.00 🔹 [id: 1 | heter: 0.11 | inst: 382 | w: 0.51]
        x_1 ≤ 0.00 🔹 [id: 2 | heter: 0.02 | inst: 192 | w: 0.26]
        x_1 > 0.00 🔹 [id: 3 | heter: 0.10 | inst: 190 | w: 0.25]
    x_0 > -0.00 🔹 [id: 4 | heter: 0.46 | inst: 368 | w: 0.49]
        x_1 ≤ -0.10 🔹 [id: 5 | heter: 0.10 | inst: 152 | w: 0.20]
        x_1 > -0.10 🔹 [id: 6 | heter: 0.14 | inst: 216 | w: 0.29]



In [19]:
y_pred = calm.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"CALM R^2 Score: {r2:.4f}")

CALM R^2 Score: 0.9441
